# Traffic Signal Violation Detection & e-Challan System (Bengaluru)

## What are we building?
An end-to-end **AI traffic enforcement demo** that:
1. Watches a traffic-junction video
2. **Detects vehicles** (car, motorcycle, bus, truck) using **YOLOv8**
3. Tracks each vehicle across frames so it's only counted **once**
4. Checks if a vehicle **crosses the stop line while the signal is RED**
5. **Detects and reads the number plate** of any violating vehicle using a second YOLOv8 model + OCR
6. Independently checks every **two-wheeler** for a **helmet**, using a third YOLOv8 model
7. Logs every violation into a **dummy SQLite database**
8. Automatically **generates an e-Challan** (traffic fine ticket) — just like Bengaluru Traffic Police's real e-Challan system

## ⚠️ Important Disclaimer (please read)
This notebook is an **educational / proof-of-concept demo**, not a production law-enforcement system:
- It uses **real sample assets** you provided: a wide-angle junction video and a zoomed-in ANPR-style
  snapshot of the same rider at the same junction (no live Bengaluru camera feed is available in this
  environment). This mirrors how real red-light-camera systems work — a **wide overview camera** to catch
  the violation, and a **separate zoomed/telephoto camera** just for reading the plate — explained in Step 2.
- The **traffic signal state (RED/GREEN)** is **simulated**, since we can't guarantee a single camera feed
  with a frame-synced, machine-readable signal in this environment. In a real deployment, the signal state
  would come directly from the junction's signal controller or a dedicated light-color sensor.
- OCR plate reads can be **wrong or incomplete** — every real-world ANPR (Automatic Number Plate Recognition) system requires a **human traffic officer to verify** the evidence photo and plate number before a challan is legally issued.
- Deploying anything like this for real enforcement requires authorization from the traffic police department, compliance with data protection/privacy laws, and integration with the official vehicle registration (Vahan/RTO) database.

## The Big Picture

```
Traffic Video
     |
YOLOv8 Vehicle Detector + Tracker  ---->  draws a box + a persistent ID around every car/bike/bus/truck
     |
     |---- Signal check -----> Virtual Stop Line + Signal State -> crossed while RED? --no--> ignore
     |                                                                   |
     |                                                                  yes --> queue for plate read
     |
     '---- Helmet check -----> (two-wheelers only) YOLOv8 Helmet Detector -> no helmet? --no--> ignore
                                                                            |
                                                                           yes --> queue for plate read
     |
YOLOv8 License Plate Detector  ---->  finds the plate region inside the vehicle's box
     |
EasyOCR                        ---->  reads the plate text (e.g. "KA01AB1234")
     |
SQLite Dummy Database          ---->  stores: plate, vehicle type, location, time, fine amount, evidence photo
     |
e-Challan Generator            ---->  prints a Bengaluru-Traffic-Police-style fine ticket
```


## Step 1: Install and Import the Libraries

- **`ultralytics`** → gives us YOLOv8, for both vehicle detection AND license plate detection.
- **`huggingface_hub`** → lets us download a YOLOv8 model that has already been **fine-tuned on a real, public license-plate dataset** (so we don't have to train our own).
- **`easyocr`** → reads text out of the cropped license-plate image (the actual "number plate recognition" part).
- **`opencv-python`** → lets Python open/read video frames and draw boxes/lines on images.
- **`pandas`** → to view our violations/database neatly as a table.


In [ ]:
# Install required libraries (only needs to run once per environment)
%pip install -q ultralytics huggingface_hub easyocr opencv-python pandas

import os
import re
import cv2
import random
import sqlite3
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from IPython.display import display, HTML

from ultralytics import YOLO
from huggingface_hub import hf_hub_download
import easyocr

print("All libraries imported successfully!")

## Step 2: Load Our Two Camera Feeds (Video + Zoomed ANPR Snapshot)

Real red-light-violation camera systems almost always use **two cameras**:
1. A **wide overview camera** that watches the whole junction, detects vehicles, and judges the stop line —
   but it's usually too far away to read a plate clearly.
2. A **zoomed-in ANPR (Automatic Number Plate Recognition) camera**, triggered once a violation is seen,
   that captures a sharp, close-up photo good enough to actually read the plate.

You've provided exactly these two real assets, so we use them directly — no internet download needed!

- **`gemini_generated_video_ce4692bc.mp4`** → our **wide junction camera**. It shows a rider waiting at the
  crosswalk/stop line, then riding forward, across the line and into the junction while cross-traffic (and a
  visible red signal light) is active — exactly the scenario we want to detect.
- **`sample_violation_image.png`** → our **zoomed ANPR camera** snapshot of the very same rider, with a sharp,
  clearly readable Indian plate (`KA03K A2878`). We'll use this in Step 5b to test our plate detector + OCR
  on a guaranteed-clear image before relying on it inside the full video pipeline.

> **To use your own footage:** just replace the two filenames below with your own video/image
> (e.g. an authorized recording from a real Bengaluru junction camera).


In [ ]:
VIDEO_PATH = "gemini_generated_video_ce4692bc.mp4"   # wide junction camera
IMAGE_PATH = "sample_violation_image.png"            # zoomed ANPR camera snapshot

assert os.path.exists(VIDEO_PATH), f"Couldn't find {VIDEO_PATH} — upload it to this notebook's folder."
assert os.path.exists(IMAGE_PATH), f"Couldn't find {IMAGE_PATH} — upload it to this notebook's folder."

# Preview the first frame of the video + the ANPR snapshot, side by side
cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS) or 24
VIDEO_FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ok, first_frame = cap.read()
cap.release()

anpr_snapshot = cv2.imread(IMAGE_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Wide junction camera (video, frame 1)")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(anpr_snapshot, cv2.COLOR_BGR2RGB))
axes[1].set_title("Zoomed ANPR camera (snapshot)")
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f"Video: {VIDEO_FRAME_COUNT} frames at {VIDEO_FPS} fps")

## Step 3: Load the Vehicle Detector (YOLOv8, pretrained on COCO)

We use **YOLOv8n** (the small/fast "nano" version), already pretrained on the **COCO dataset**
(a large, real-world dataset of everyday photos). It already knows how to detect `car`, `motorcycle`,
`bus`, and `truck` — we don't need to train anything ourselves!

We also set up a **fine amount for each vehicle type**, based on Bengaluru Traffic Police's published
red-light-violation fines (indicative figures, for this educational demo — always check the official
Bengaluru Traffic Police / Karnataka Transport Department site for current amounts):

| COCO class | Vehicle type | Fine for jumping red signal |
|---|---|---|
| `motorcycle` | Two-wheeler | ₹1,000 |
| `car` | Four-wheeler | ₹2,000 |
| `bus`, `truck` | Heavy vehicle | ₹5,000 |


In [ ]:
# Load the pretrained YOLOv8 vehicle detector
vehicle_model = YOLO("yolov8n.pt")

# COCO class id -> (readable vehicle type, fine amount in INR)
VEHICLE_FINES = {
    2: ("Car / Four-wheeler", 2000),
    3: ("Motorcycle / Two-wheeler", 1000),
    5: ("Bus / Heavy Vehicle", 5000),
    7: ("Truck / Heavy Vehicle", 5000),
}

print("Vehicle detector ready. Watching for:", [v[0] for v in VEHICLE_FINES.values()])

## Step 4: Load the License Plate Detector (YOLOv8, fine-tuned on a real plate dataset)

A generic COCO model doesn't know what a "license plate" looks like — COCO has no such class.
So we use a **second YOLOv8 model**, this time one that was **fine-tuned (further trained)**
specifically on a real-world, public **license-plate dataset**
(`Koushim/yolov8-license-plate-detection` on Hugging Face Hub).

**Fine-tuning** simply means: start from a model that already understands general images,
then train it a bit more on a focused dataset (here, thousands of real photos of number plates)
so it becomes a specialist at finding plates.


In [ ]:
# Download the fine-tuned license-plate weights from Hugging Face Hub
plate_weights_path = hf_hub_download(
    repo_id="Koushim/yolov8-license-plate-detection",
    filename="best.pt"
)

plate_model = YOLO(plate_weights_path)
print("License plate detector ready!")

## Step 4b: Load the Helmet Detector (YOLOv8, fine-tuned on a real helmet dataset)

Just like a generic COCO model doesn't know what a license plate looks like, it also doesn't know
what a "helmet" is. So we load a **third YOLOv8 model**, this one fine-tuned on a real-world dataset
of riders with and without helmets (`iam-tsr/yolov8n-helmet-detection` on Hugging Face Hub, classes
`helmet` / `no_helmet`).

This lets us flag a very common, serious real-world offense: **riding a two-wheeler without a
helmet** — punishable under **Section 129 of the Motor Vehicles Act**, with a fine of **₹1,000** in
Bengaluru (2025 rate).


In [ ]:
# Download the fine-tuned helmet-detection weights from Hugging Face Hub
helmet_weights_path = hf_hub_download(
    repo_id="iam-tsr/yolov8n-helmet-detection",
    filename="best.pt"
)

helmet_model = YOLO(helmet_weights_path)
HELMET_FINE_AMOUNT = 1000  # INR, Sec 129 Motor Vehicles Act (Bengaluru, 2025 rate)

print("Helmet detector ready! Classes:", helmet_model.names)

## Step 5: Read the Plate Text with OCR (EasyOCR)

**OCR (Optical Character Recognition)** is the technology that turns an *image* of text into
*actual, usable text characters* — exactly what we need to turn a cropped plate photo into
a string like `"KA01AB1234"`.

A naive "crop + OCR" pipeline often gets individual characters wrong — especially on small,
**blurry**, low-contrast plate crops pulled out of a video frame. For example, on this very
project, OCR once read a real plate `KA03KA2878` as `KAOJK42878`. Look closely at *where* it went
wrong: `0→O`, `3→J`, `A→4` — these are all **classic OCR digit/letter look-alike mix-ups**, made
worse by blur smearing the exact shape of each character.

So our function does **6 things**, not 3:
1. Runs the **plate detector** on the cropped vehicle image to find the plate's exact location.
2. Crops just that plate region.
3. **Sharpens** the crop — this is the key fix for blur. We:
   - **Upscale** it first (OCR reads big text far more reliably than tiny text).
   - **Denoise without blurring edges** using a *bilateral filter* (unlike a normal blur, it smooths
     flat areas like the plate's background while keeping character edges crisp).
   - Apply **unsharp masking** — the classic sharpening trick: blur a copy of the image, then push
     the original *further away* from that blurry version. This exaggerates edges exactly where
     blur had softened them.
   - Apply **CLAHE** (adaptive local contrast) instead of a single global contrast boost, so glare
     or shadow on one part of the plate doesn't wash out the rest.
4. **Also tries a binarized (pure black/white) version** of the sharpened crop — sometimes this
   helps OCR, sometimes it hurts (jagged edges can confuse it), so we run OCR on **both** versions
   and automatically keep whichever gives the better result.
5. Runs **EasyOCR** with an `allowlist` restricted to `A-Z0-9`, then **sorts the detected text top-to-
   bottom, then left-to-right** before joining it — this matters a lot for real Indian two-wheeler
   plates, which are usually printed on **two lines** (e.g. `KA03K` on top, `A2878` below)!
6. **Corrects common OCR mix-ups** using the plate's *known, fixed structure* — Indian plates always
   follow `LL DD LL DDDD` (2 letters, 2 digits, 1-2 letters, 4 digits). If a position that *must* be a
   letter contains a digit (or vice versa), we swap it for its most common look-alike
   (`0↔O`, `1↔I`, `2↔Z`, `3↔J`, `4↔A`, `5↔S`, `6↔G`, `8↔B`).

Finally, we check the corrected text against a general **Indian number plate pattern**
(`SS` + 2 digits + 1-2 letters + 4 digits, e.g. `KA01AB1234`) to flag plates that still may need
a human to double-check — OCR (even sharpened and corrected) is never 100% perfect!


In [ ]:
# EasyOCR reader (English)
ocr_reader = easyocr.Reader(['en'], gpu=False)

# General Indian number-plate pattern, e.g. KA01AB1234
INDIAN_PLATE_PATTERN = re.compile(r'^[A-Z]{2}\d{2}[A-Z]{1,2}\d{4}$')

# OCR's most common digit<->letter look-alike mix-ups
DIGIT_TO_LETTER = {'0': 'O', '1': 'I', '2': 'Z', '3': 'J', '4': 'A', '5': 'S', '6': 'G', '8': 'B'}
LETTER_TO_DIGIT = {letter: digit for digit, letter in DIGIT_TO_LETTER.items()}


def sharpen_plate_crop(plate_crop):
    # Fix BLUR and LOW CONTRAST before OCR ever sees the image — this matters far more
    # than anything we could do to the text afterwards.
    height, width = plate_crop.shape[:2]

    # 1) Upscale so the plate is roughly 220px tall (OCR reads big text more reliably)
    scale = max(1, 220 // max(height, 1))
    if scale > 1:
        plate_crop = cv2.resize(plate_crop, (width * scale, height * scale), interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)

    # 2) Denoise WITHOUT blurring edges (bilateral filter keeps character strokes crisp
    #    while smoothing flat areas like the plate's background)
    denoised = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

    # 3) Unsharp masking: blur a copy, then push the ORIGINAL further away from that
    #    blur -> edges that blur had softened get exaggerated back into focus.
    blurred = cv2.GaussianBlur(denoised, (0, 0), sigmaX=3)
    sharpened = cv2.addWeighted(denoised, 1.5, blurred, -0.5, 0)

    # 4) CLAHE: adaptive LOCAL contrast, so glare/shadow on one part of the plate
    #    doesn't wash out the rest (better than a single global contrast boost)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(sharpened)
    return enhanced


def binarize_plate_crop(enhanced_gray):
    # Pure black/white version using Otsu's method (auto-picks the best threshold).
    # A tiny morphological "close" smooths out jagged single-pixel noise along edges.
    _, binary = cv2.threshold(enhanced_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)


def ocr_read_sorted(image, conf_threshold=0.3):
    # Run EasyOCR, then sort detected text TOP-TO-BOTTOM then LEFT-TO-RIGHT before
    # joining it. This matters a lot for real Indian two-wheeler plates, which are
    # usually printed on TWO LINES (e.g. "KA03K" above "A2878") — without sorting,
    # the lines could get joined in the wrong order.
    results = ocr_reader.readtext(image, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    fragments = [(box, text, conf) for (box, text, conf) in results if conf > conf_threshold]
    fragments.sort(key=lambda item: (min(p[1] for p in item[0]), min(p[0] for p in item[0])))

    raw_text = "".join(text for (_, text, _) in fragments)
    avg_conf = (sum(conf for (_, _, conf) in fragments) / len(fragments)) if fragments else 0.0
    return raw_text, avg_conf


def correct_plate_text(raw_text):
    # Fix OCR digit<->letter mix-ups using the KNOWN, fixed structure of an Indian plate:
    #   LL DD LL DDDD  (10 chars, 2-letter series)  or  LL DD L DDDD  (9 chars, 1-letter series)
    if len(raw_text) == 10:
        letter_positions, digit_positions = [0, 1, 4, 5], [2, 3, 6, 7, 8, 9]
    elif len(raw_text) == 9:
        letter_positions, digit_positions = [0, 1, 4], [2, 3, 5, 6, 7, 8]
    else:
        return raw_text  # unexpected length -> leave as-is (still flagged for manual review)

    chars = list(raw_text)
    for i in letter_positions:
        if chars[i].isdigit():
            chars[i] = DIGIT_TO_LETTER.get(chars[i], chars[i])
    for i in digit_positions:
        if chars[i].isalpha():
            chars[i] = LETTER_TO_DIGIT.get(chars[i], chars[i])
    return "".join(chars)


def detect_and_read_plate(vehicle_crop):
    # Given a cropped vehicle image, find + read its license plate.
    # Returns (plate_text, plate_is_valid_format, plate_crop_image)
    if vehicle_crop is None or vehicle_crop.size == 0:
        return "UNREADABLE", False, None

    results = plate_model(vehicle_crop, verbose=False)
    boxes = results[0].boxes

    if len(boxes) == 0:
        return "UNREADABLE", False, None

    # Use the plate detection with the highest confidence
    best_idx = int(boxes.conf.argmax())
    px1, py1, px2, py2 = map(int, boxes.xyxy[best_idx])
    plate_crop = vehicle_crop[max(0, py1):py2, max(0, px1):px2]

    if plate_crop.size == 0:
        return "UNREADABLE", False, None

    sharpened_crop = sharpen_plate_crop(plate_crop)
    binary_crop = binarize_plate_crop(sharpened_crop)

    # Try OCR on both the sharpened grayscale AND the binarized version, and automatically
    # keep whichever read matches the expected plate format, or otherwise has higher confidence.
    best_text, best_valid, best_conf = "UNREADABLE", False, -1.0
    for candidate in (sharpened_crop, binary_crop):
        raw_text, avg_conf = ocr_read_sorted(candidate)
        clean_text = re.sub(r'[^A-Z0-9]', '', raw_text.upper())
        if not clean_text:
            continue

        corrected_text = correct_plate_text(clean_text)
        is_valid = bool(INDIAN_PLATE_PATTERN.match(corrected_text))

        is_better = (is_valid and not best_valid) or (is_valid == best_valid and avg_conf > best_conf)
        if is_better:
            best_text, best_valid, best_conf = corrected_text, is_valid, avg_conf

    return best_text, best_valid, plate_crop


print("Plate reading function ready!")

## Step 5b: Quick Test — Read the Plate from Our Zoomed ANPR Snapshot

Before trusting this pipeline on a full video, let's sanity-check it on something close to our worst
case: a small, slightly blurry plate crop. This is exactly how you'd debug a real ANPR system — test
the plate-reading sub-pipeline in isolation, and **look at what OCR is actually seeing**.


In [ ]:
test_plate_text, test_plate_valid, test_plate_crop = detect_and_read_plate(anpr_snapshot)

print(f"Plate read: '{test_plate_text}'  |  matches Indian format: {test_plate_valid}")

if test_plate_crop is not None:
    test_sharpened = sharpen_plate_crop(test_plate_crop)
    test_binary = binarize_plate_crop(test_sharpened)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].imshow(cv2.cvtColor(test_plate_crop, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Crop")
    axes[1].imshow(test_sharpened, cmap='gray')
    axes[1].set_title("Sharpened + CLAHE")
    axes[2].imshow(test_binary, cmap='gray')
    axes[2].set_title("Binarized (Otsu)")
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f"Plate read -> '{test_plate_text}'")
    plt.tight_layout()
    plt.show()

## Step 5c: Define and Test the Helmet-Check Function

Our helper function:
1. Runs the **helmet detector** on a cropped two-wheeler image.
2. Picks the single most confident detection.
3. Returns a simple status: `"HELMET"`, `"NO_HELMET"`, or `"UNKNOWN"` (if no head/helmet could be
   found at all — e.g. the rider is too far away or turned away from the camera).

Different models label their classes slightly differently (`no_helmet`, `No-Helmet`, `without_helmet`...),
so instead of hardcoding one exact spelling, we just check whether the word "no" appears in the
predicted class name.

Let's test it on our zoomed ANPR snapshot — look closely at the earlier image and you'll notice the
rider's hand is on his bare head, no helmet in sight!


In [ ]:
def check_helmet(vehicle_crop):
    # Given a cropped TWO-WHEELER image, check whether the rider is wearing a helmet.
    # Returns (status, confidence, head_crop_image) where status is
    # "HELMET", "NO_HELMET", or "UNKNOWN" (nothing detected).
    if vehicle_crop is None or vehicle_crop.size == 0:
        return "UNKNOWN", 0.0, None

    results = helmet_model(vehicle_crop, verbose=False)
    boxes = results[0].boxes

    if len(boxes) == 0:
        return "UNKNOWN", 0.0, None

    # Use the single most confident detection
    best_idx = int(boxes.conf.argmax())
    confidence = float(boxes.conf[best_idx])
    class_name = helmet_model.names[int(boxes.cls[best_idx])]

    hx1, hy1, hx2, hy2 = map(int, boxes.xyxy[best_idx])
    head_crop = vehicle_crop[max(0, hy1):hy2, max(0, hx1):hx2]

    status = "NO_HELMET" if "no" in class_name.lower() else "HELMET"
    return status, confidence, head_crop


test_helmet_status, test_helmet_conf, test_head_crop = check_helmet(anpr_snapshot)
print(f"Helmet status: {test_helmet_status}  (confidence: {test_helmet_conf:.2f})")

if test_head_crop is not None:
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(test_head_crop, cv2.COLOR_BGR2RGB))
    plt.title(f"Detected head region -> {test_helmet_status}")
    plt.axis('off')
    plt.show()

## Step 6: Define the Virtual Stop Line and the Traffic Signal State

Two key ingredients for detecting a "red-light jump":

1. **Stop line** — an imaginary horizontal line drawn across the road, at the near edge of the
   crosswalk where the rider is initially waiting. We place it at **88% down the frame**
   (`STOP_LINE_FRACTION`). Our video's camera looks *forward into the junction* from just behind the
   stop line, so a vehicle that has crossed it moves **farther away and higher up in the frame**
   (its box gets smaller and rises) rather than closer/bigger — the opposite of a camera facing
   oncoming traffic. We capture that with `CROSSING_DIRECTION = "away_from_camera"`.
   **In a real deployment, both the line position and this direction would be calibrated once per camera.**

2. **Signal state** — whether the light is RED or GREEN right now. Our video is a short, purpose-made
   "violation" clip (it even briefly shows a real red signal light mid-clip!), and it's edited together
   from a couple of different camera angles rather than one continuous static shot. Rather than
   fragile frame-perfect calibration across every angle, we keep it simple and honest: we treat almost
   the **entire clip as RED** (`SIGNAL_SWITCH_FRAME`), matching the story the video actually tells —
   a rider who doesn't stop. For a real fixed CCTV camera, you'd instead drive the signal state from
   the junction's real controller, or from `detect_signal_color_from_roi()` below.

   **Bonus (for real deployments):** `detect_signal_color_from_roi()` below shows how you *could*
   detect the actual signal color directly from a small region of the camera frame (using classic
   color-thresholding in OpenCV), if the traffic light itself is visible in the camera's view.


In [ ]:
STOP_LINE_FRACTION = 0.88        # stop line sits near the crosswalk, 88% down the frame
CROSSING_DIRECTION = "away_from_camera"  # this camera faces INTO the junction
SIGNAL_SWITCH_FRAME = 230        # RED for almost the whole clip (it's a dedicated violation demo)


def get_signal_state(frame_idx):
    # Simulated signal: RED for the first part of the clip, then GREEN.
    # Replace with a real signal-controller feed in production.
    return "RED" if frame_idx <= SIGNAL_SWITCH_FRAME else "GREEN"


def has_crossed_stop_line(y2, stop_line_y):
    # Whether a box's bottom edge (y2) is already past the stop line, accounting for
    # which way this particular camera faces relative to the junction.
    if CROSSING_DIRECTION == "away_from_camera":
        return y2 < stop_line_y   # vehicle has moved further in and appears higher/smaller
    return y2 > stop_line_y       # vehicle is approaching the camera and appears lower/larger


def detect_signal_color_from_roi(frame, roi):
    # BONUS: detect the real signal color from a region of interest (x1, y1, x2, y2)
    # in the camera frame, if the traffic light is visible. Not used by default in this demo.
    x1, y1, x2, y2 = roi
    patch = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)

    red_mask = cv2.inRange(hsv, (0, 100, 100), (10, 255, 255)) | cv2.inRange(hsv, (160, 100, 100), (179, 255, 255))
    green_mask = cv2.inRange(hsv, (40, 70, 70), (80, 255, 255))

    red_px = cv2.countNonZero(red_mask)
    green_px = cv2.countNonZero(green_mask)

    if red_px > green_px and red_px > 20:
        return "RED"
    elif green_px > red_px and green_px > 20:
        return "GREEN"
    return "UNKNOWN"


print("Stop line fraction:", STOP_LINE_FRACTION)

## Step 7: The Main Violation-Detection Loop

This is where everything comes together. We use YOLOv8's **built-in object tracker**
(`model.track(...)`), which gives every vehicle a **persistent ID** that stays the same across
frames (powered by the ByteTrack algorithm) — this is how we avoid fining the same vehicle
more than once for the same crossing!

For every frame, we:
1. Work out the current **signal state** (RED or GREEN).
2. Draw the **stop line** (red if signal is RED, green if GREEN) for visualization.
3. For every tracked vehicle, check: *is its bottom edge past the stop line right now (accounting
   for which way the camera faces), while the signal is RED?*
4. If yes **and** we haven't already ticketed this track ID → crop the vehicle, detect + read its
   plate, save an evidence photo, and record the violation. Each track ID is only ever ticketed once,
   even though it may stay past the line for many more frames.
5. **Independently** (regardless of the signal state), for every tracked **two-wheeler** we check
   once whether the rider is wearing a helmet — just like a real helmet-detection camera would run
   as its own, separate analytics feed. No helmet → its own violation + challan.

We cap processing at `MAX_FRAMES` to keep this demo quick to run.


In [ ]:
MAX_FRAMES = VIDEO_FRAME_COUNT  # process the whole clip (~10 seconds)

os.makedirs("evidence", exist_ok=True)

ticketed_ids = set()         # track_ids already fined for a signal violation
helmet_checked_ids = set()   # track_ids already checked for a helmet (once each)
violations = []              # collected violation records (both kinds)

frame_idx = 0

for result in vehicle_model.track(
    source=VIDEO_PATH,
    stream=True,
    persist=True,
    tracker="bytetrack.yaml",
    classes=list(VEHICLE_FINES.keys()),
    verbose=False
):
    frame_idx += 1
    if frame_idx > MAX_FRAMES:
        break

    frame = result.orig_img.copy()
    height, width = frame.shape[:2]
    stop_line_y = int(height * STOP_LINE_FRACTION)
    signal_state = get_signal_state(frame_idx)

    # Draw the stop line + current signal state
    line_color = (0, 0, 255) if signal_state == "RED" else (0, 200, 0)
    cv2.line(frame, (0, stop_line_y), (width, stop_line_y), line_color, 3)
    cv2.putText(frame, f"SIGNAL: {signal_state}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, line_color, 2)

    if result.boxes.id is not None:
        for box in result.boxes:
            track_id = int(box.id[0])
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            vehicle_crop = frame[max(0, y1):y2, max(0, x1):x2]

            # Is this vehicle's bottom edge past the stop line right now?
            past_line = has_crossed_stop_line(y2, stop_line_y)

            box_color = (0, 200, 0)

            # --- Check 1: red-light / stop-line violation ---
            if signal_state == "RED" and past_line and track_id not in ticketed_ids:
                box_color = (0, 0, 255)
                ticketed_ids.add(track_id)

                plate_text, plate_valid, plate_crop = detect_and_read_plate(vehicle_crop)

                evidence_path = f"evidence/violation_track{track_id}_frame{frame_idx}.jpg"
                cv2.imwrite(evidence_path, frame)

                vehicle_type, fine_amount = VEHICLE_FINES[cls_id]
                violations.append({
                    "track_id": track_id,
                    "vehicle_type": vehicle_type,
                    "fine_amount": fine_amount,
                    "plate_number": plate_text,
                    "plate_valid_format": plate_valid,
                    "frame_number": frame_idx,
                    "evidence_image": evidence_path,
                    "violation_type": "Disobeying Traffic Signal / Jumping Red Light (Sec 119/177, Motor Vehicles Act)",
                })
                print(f"Frame {frame_idx}: SIGNAL VIOLATION caught -> track {track_id}, "
                      f"{vehicle_type}, plate reads as '{plate_text}'")

            # --- Check 2: helmet check (two-wheelers only, independent of signal state) ---
            if cls_id == 3 and track_id not in helmet_checked_ids:
                helmet_checked_ids.add(track_id)
                helmet_status, helmet_conf, _ = check_helmet(vehicle_crop)

                if helmet_status == "NO_HELMET":
                    box_color = (0, 0, 255)
                    plate_text_h, plate_valid_h, _ = detect_and_read_plate(vehicle_crop)

                    evidence_path_h = f"evidence/nohelmet_track{track_id}_frame{frame_idx}.jpg"
                    cv2.imwrite(evidence_path_h, frame)

                    violations.append({
                        "track_id": track_id,
                        "vehicle_type": VEHICLE_FINES[cls_id][0],
                        "fine_amount": HELMET_FINE_AMOUNT,
                        "plate_number": plate_text_h,
                        "plate_valid_format": plate_valid_h,
                        "frame_number": frame_idx,
                        "evidence_image": evidence_path_h,
                        "violation_type": "Riding Without Helmet (Sec 129, Motor Vehicles Act)",
                    })
                    print(f"Frame {frame_idx}: NO HELMET caught -> track {track_id}, "
                          f"plate reads as '{plate_text_h}'")

            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
            cv2.putText(frame, f"ID {track_id}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

print(f"\nFinished scanning {frame_idx} frames. Violations detected: {len(violations)}")

## Step 7b: Look at One Violation's Evidence Photo

Let's view the evidence photo for the first detected violation (if any), so we can see
exactly what the system "saw" when it issued the ticket — just like a real traffic camera snapshot.


In [ ]:
if violations:
    first_violation = violations[0]
    evidence_img = cv2.imread(first_violation["evidence_image"])
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(evidence_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Evidence photo — Track ID {first_violation['track_id']} — "
              f"Plate: {first_violation['plate_number']}")
    plt.axis('off')
    plt.show()
else:
    print("No violations were detected in this demo run — try increasing MAX_FRAMES, "
          "or adjusting SIGNAL_SWITCH_FRAME/STOP_LINE_FRACTION, and re-run Step 7.")

## Step 8: Create the Dummy SQLite Database

**SQLite** is a lightweight, file-based database — perfect for a demo like this because it needs
**no separate server**, just a single `.db` file on disk.

We create one table, `challans`, to store every traffic violation ticket we generate:

| Column | Meaning |
|---|---|
| `challan_id` | Unique ticket number |
| `plate_number` | Number plate read by OCR |
| `plate_verified` | Did the plate match a valid Indian plate format? |
| `vehicle_type` | Car / Motorcycle / Bus / Truck |
| `violation_type` | Description of the offense |
| `location` | Junction/camera location |
| `violation_datetime` | When the violation happened |
| `fine_amount` | Fine in ₹ |
| `evidence_image` | Path to the evidence photo |
| `status` | PAID / UNPAID |


In [ ]:
DB_PATH = "traffic_violations.db"
CAMERA_LOCATION = "Silk Board Junction, Bengaluru"  # illustrative label only — our demo footage isn't actually from here

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS challans (
    challan_id TEXT PRIMARY KEY,
    plate_number TEXT,
    plate_verified INTEGER,
    vehicle_type TEXT,
    violation_type TEXT,
    location TEXT,
    violation_datetime TEXT,
    fine_amount INTEGER,
    evidence_image TEXT,
    status TEXT
)
''')
conn.commit()

print("Database ready at:", DB_PATH)

## Step 9: Log Every Violation as a Challan in the Database

For each violation we caught in Step 7 — whether it's a **signal jump** or a **no-helmet** offense,
each one already carries its own `violation_type` and `fine_amount` — we:
- Generate a unique **challan number** (e.g. `KATP483920`)
- Work out an approximate **real-world timestamp** for the violation, using the video's frame
  number and FPS (frames per second)
- Insert a new row into the `challans` table, with status `"UNPAID"`


In [ ]:
base_time = datetime.now()

def make_challan_id():
    return f"KATP{random.randint(100000, 999999)}"

for v in violations:
    challan_id = make_challan_id()
    seconds_into_video = v["frame_number"] / VIDEO_FPS
    violation_time = base_time + timedelta(seconds=seconds_into_video)

    cursor.execute(
        '''INSERT INTO challans
           (challan_id, plate_number, plate_verified, vehicle_type, violation_type,
            location, violation_datetime, fine_amount, evidence_image, status)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
        (
            challan_id,
            v["plate_number"],
            int(v["plate_valid_format"]),
            v["vehicle_type"],
            v["violation_type"],
            CAMERA_LOCATION,
            violation_time.strftime("%Y-%m-%d %H:%M:%S"),
            v["fine_amount"],
            v["evidence_image"],
            "UNPAID",
        )
    )
    v["challan_id"] = challan_id
    v["location"] = CAMERA_LOCATION
    v["violation_datetime"] = violation_time.strftime("%Y-%m-%d %H:%M:%S")
    v["status"] = "UNPAID"

conn.commit()
print(f"Logged {len(violations)} challan(s) into the database.")

## Step 10: Generate the e-Challan

A **challan** is the Indian term for an official traffic fine ticket. Let's build a function that
renders one as a nicely formatted HTML notice — similar in spirit to Bengaluru Traffic Police's
real e-Challan portal — and save a copy of each one to the `challans/` folder.


In [ ]:
os.makedirs("challans", exist_ok=True)


def generate_challan_html(v):
    verified_note = (
        "✅ Plate number matched a standard Indian plate format."
        if v["plate_valid_format"] else
        "⚠️ Plate could not be fully verified by OCR — pending manual review by a traffic officer."
    )
    return f'''
    <div style="border:3px solid #b30000; border-radius:10px; padding:20px; width:480px;
                font-family:Arial, sans-serif; background:#fff8f8;">
      <h2 style="text-align:center; color:#b30000; margin-bottom:0;">BENGALURU CITY TRAFFIC POLICE</h2>
      <p style="text-align:center; margin-top:2px; color:#555;">e-Challan - Traffic Violation Notice</p>
      <hr style="border:1px solid #b30000;">
      <table style="width:100%; font-size:14px;">
        <tr><td><b>Challan No:</b></td><td>{v['challan_id']}</td></tr>
        <tr><td><b>Date &amp; Time:</b></td><td>{v['violation_datetime']}</td></tr>
        <tr><td><b>Location:</b></td><td>{v['location']}</td></tr>
        <tr><td><b>Vehicle Number:</b></td>
            <td style="font-size:18px; font-weight:bold; color:#b30000;">{v['plate_number']}</td></tr>
        <tr><td><b>Vehicle Type:</b></td><td>{v['vehicle_type']}</td></tr>
        <tr><td><b>Violation:</b></td><td>{v['violation_type']}</td></tr>
        <tr><td><b>Fine Amount:</b></td><td style="font-size:16px; font-weight:bold;">Rs. {v['fine_amount']}</td></tr>
        <tr><td><b>Status:</b></td>
            <td><span style="color:white; background:#b30000; padding:2px 8px; border-radius:4px;">{v['status']}</span></td></tr>
      </table>
      <hr style="border:1px dashed #b30000;">
      <p style="font-size:12px; color:#555;">
        {verified_note}<br>
        Pay within 15 days to avoid a late fee. Visit the official Bengaluru Traffic Police
        e-Challan portal or your nearest traffic kiosk to pay and view evidence photos.
      </p>
    </div>
    '''


for v in violations:
    html = generate_challan_html(v)
    with open(f"challans/{v['challan_id']}.html", "w", encoding="utf-8") as f:
        f.write(html)
    display(HTML(html))

## Step 11: View All Issued Challans

Finally, let's pull everything back out of the database and view it as a clean table —
exactly what a traffic police dashboard might show.


In [ ]:
challans_df = pd.read_sql_query("SELECT * FROM challans", conn)
challans_df

## Wrap-Up: From Demo to Real-World System

This notebook demonstrates the full pipeline end-to-end, but turning it into a real traffic
enforcement tool would require a lot more:

- **Real signal-state input** — directly from the junction's signal controller, not a timer.
- **Multiple camera angles** — to confirm a vehicle truly crossed the line, from more than one view.
- **Better plate OCR** — handling glare, motion blur, dirty plates, and non-standard fonts/state codes.
- **Human review** — a traffic officer should always verify the evidence photo and plate reading
  before a challan is legally issued and sent.
- **Integration with the Vahan/RTO vehicle registration database** — to look up the registered
  owner's name and address so the challan can actually be delivered.
- **Data privacy & legal compliance** — proper authorization, data retention limits, and audit
  trails are required before deploying any camera-based enforcement system.
- **Robustness** — handling night-time, rain, fog, and heavy traffic congestion reliably.

Even so, the core ideas used here — **YOLOv8 object detection + tracking, a fine-tuned detector
for a specialized object (plates), OCR, and a simple database-backed workflow** — are exactly
the building blocks real-world ANPR and traffic-enforcement systems are built on.
